# Using Multiple Regression to predict the outcome of tennis matches
## Authors: Samuel Alkire, Ryan Sun, Mithun Rameshkumar



Here we upload and clean the dataset as follows:


1.   The Dataset comes from the ATP and spans from 1968 to 2024
2.   We then sort the data by date so it is in chronological order
3. Keep only the columns that are needed for our analysis
4. Clean any rows with incomplete data



In [8]:
import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

#Load Data
df = pd.read_csv("/content/combined_atp_1968_2024.csv")

#Sort chronologically
df['tourney_date'] = pd.to_datetime(df['tourney_date'], format="%Y%m%d", errors="coerce")
df = df.sort_values('tourney_date').reset_index(drop=True)

# Keep only rows with all the stats we need
needed_cols = [
    'winner_id','loser_id',
    'winner_rank','loser_rank',
    'winner_age','loser_age',
    'winner_ht','loser_ht',
    'surface',
    'w_ace','w_df','w_svpt','w_1stIn','w_1stWon','w_2ndWon','w_bpSaved','w_bpFaced','w_SvGms',
    'l_ace','l_df','l_svpt','l_1stIn','l_1stWon','l_2ndWon','l_bpSaved','l_bpFaced','l_SvGms'
]
df = df.dropna(subset=needed_cols)
df = df[(df['w_svpt'] > 0) & (df['l_svpt'] > 0)]

print("After cleaning:", df.shape)


FileNotFoundError: [Errno 2] No such file or directory: '/content/combined_atp_1968_2024.csv'

In [ ]:
def make_stat_dict():
    return {
        'wins': 0,
        'losses': 0,

        # serve
        'aces': 0,
        'dfs': 0,
        'svpt': 0,
        'first_in': 0,
        'first_won': 0,
        'second_won': 0,
        'bp_saved': 0,
        'bp_faced': 0,
        'sv_gms': 0,

        # return (from opponent serve)
        'ret_pts_won': 0,
        'ret_pts_total': 0,
        'ret_bp_conv': 0,
        'ret_bp_opps': 0,
        'ret_gms_won': 0,
        'ret_gms_played': 0,

        # total points (serve + return)
        'total_pts_won': 0,
        'total_pts_total': 0,
    }

# career-wide stats per player
overall_stats = defaultdict(make_stat_dict)
# per-surface stats per player
surface_stats = defaultdict(lambda: defaultdict(make_stat_dict))


In [ ]:
def safe_ratio(num, den):
    return float(num) / den if den and den > 0 else -1.0

def compute_player_features(player_id, surface, rank, age, height):
    o = overall_stats[player_id]
    s = surface_stats[player_id][surface]

    feats = {}

    #current rank and bio
    feats['rank'] = rank if not np.isnan(rank) else 5000  # high for unranked
    feats['age'] = age
    feats['height'] = height

    #win/loss indices
    feats['overall_wl']  = safe_ratio(o['wins'], o['wins'] + o['losses'])
    feats['surface_wl']  = safe_ratio(s['wins'], s['wins'] + s['losses'])

    # Double faults/aces by surface
    feats['df_ace_ratio_surface'] = safe_ratio(s['dfs'], s['aces']) if s['aces'] > 0 else -1.0

    # Serving on surface
    feats['fs_pct_surface']         = safe_ratio(s['first_in'], s['svpt'])
    feats['fs_pts_won_pct_surface'] = safe_ratio(s['first_won'], s['first_in'])
    feats['ss_pts_won_pct_surface'] = safe_ratio(s['second_won'], s['svpt'] - s['first_in'])

    feats['sv_pts_won_pct_surface'] = safe_ratio(
        s['first_won'] + s['second_won'], s['svpt']
    )

    # Return on surface
    feats['ret_pts_won_pct_surface'] = safe_ratio(
        s['ret_pts_won'], s['ret_pts_total']
    )

    # Break points
    feats['bp_saved_pct_surface'] = safe_ratio(s['bp_saved'], s['bp_faced'])
    feats['bp_conv_pct_surface']  = safe_ratio(s['ret_bp_conv'], s['ret_bp_opps'])

    # Total points
    feats['total_pts_won_pct_surface'] = safe_ratio(
        s['total_pts_won'], s['total_pts_total']
    )

    # return-game win %
    feats['ret_gm_won_pct_surface'] = safe_ratio(
        s['ret_gms_won'], s['ret_gms_played']
    )

    return feats


In [ ]:
def update_player_after_match(row):
    # Global decay factor
    DECAY = 0.99
    surface = row['surface']
    w = int(row['winner_id'])
    l = int(row['loser_id'])

    #Applying decay function
    def apply_decay(stats):
        for key in stats:
            stats[key] *= DECAY

    # decay winner stats
    apply_decay(overall_stats[w])
    apply_decay(surface_stats[w][surface])

    # decay loser stats
    apply_decay(overall_stats[l])
    apply_decay(surface_stats[l][surface])

    w_o = overall_stats[w]
    w_s = surface_stats[w][surface]

    w_o['wins'] += 1
    w_s['wins'] += 1

    #if winner is serving
    for d_o, d_s, prefix in [(w_o, w_s, 'w')]:
        d_o['aces']       += row[f'{prefix}_ace']
        d_o['dfs']        += row[f'{prefix}_df']
        d_o['svpt']       += row[f'{prefix}_svpt']
        d_o['first_in']   += row[f'{prefix}_1stIn']
        d_o['first_won']  += row[f'{prefix}_1stWon']
        d_o['second_won'] += row[f'{prefix}_2ndWon']
        d_o['bp_saved']   += row[f'{prefix}_bpSaved']
        d_o['bp_faced']   += row[f'{prefix}_bpFaced']
        d_o['sv_gms']     += row[f'{prefix}_SvGms']

        d_s['aces']       += row[f'{prefix}_ace']
        d_s['dfs']        += row[f'{prefix}_df']
        d_s['svpt']       += row[f'{prefix}_svpt']
        d_s['first_in']   += row[f'{prefix}_1stIn']
        d_s['first_won']  += row[f'{prefix}_1stWon']
        d_s['second_won'] += row[f'{prefix}_2ndWon']
        d_s['bp_saved']   += row[f'{prefix}_bpSaved']
        d_s['bp_faced']   += row[f'{prefix}_bpFaced']
        d_s['sv_gms']     += row[f'{prefix}_SvGms']

    #if loser is serving
    l_o = overall_stats[l]
    l_s = surface_stats[l][surface]

    l_o['losses'] += 1
    l_s['losses'] += 1

    for d_o, d_s, prefix in [(l_o, l_s, 'l')]:
        d_o['aces']       += row[f'{prefix}_ace']
        d_o['dfs']        += row[f'{prefix}_df']
        d_o['svpt']       += row[f'{prefix}_svpt']
        d_o['first_in']   += row[f'{prefix}_1stIn']
        d_o['first_won']  += row[f'{prefix}_1stWon']
        d_o['second_won'] += row[f'{prefix}_2ndWon']
        d_o['bp_saved']   += row[f'{prefix}_bpSaved']
        d_o['bp_faced']   += row[f'{prefix}_bpFaced']
        d_o['sv_gms']     += row[f'{prefix}_SvGms']

        d_s['aces']       += row[f'{prefix}_ace']
        d_s['dfs']        += row[f'{prefix}_df']
        d_s['svpt']       += row[f'{prefix}_svpt']
        d_s['first_in']   += row[f'{prefix}_1stIn']
        d_s['first_won']  += row[f'{prefix}_1stWon']
        d_s['second_won'] += row[f'{prefix}_2ndWon']
        d_s['bp_saved']   += row[f'{prefix}_bpSaved']
        d_s['bp_faced']   += row[f'{prefix}_bpFaced']
        d_s['sv_gms']     += row[f'{prefix}_SvGms']

    #if winner is returning
    w_ret_pts_won   = row['l_svpt'] - row['l_1stWon'] - row['l_2ndWon']
    w_ret_pts_total = row['l_svpt']
    w_bp_opps       = row['l_bpFaced']
    w_bp_conv       = row['l_bpFaced'] - row['l_bpSaved']
    w_ret_gms_played = row['l_SvGms']
    w_ret_gms_won    = w_bp_conv

    for d_o, d_s in [(w_o, w_s)]:
        d_o['ret_pts_won']   += w_ret_pts_won
        d_o['ret_pts_total'] += w_ret_pts_total
        d_o['ret_bp_conv']   += w_bp_conv
        d_o['ret_bp_opps']   += w_bp_opps
        d_o['ret_gms_played'] += w_ret_gms_played
        d_o['ret_gms_won']   += w_ret_gms_won

        d_s['ret_pts_won']   += w_ret_pts_won
        d_s['ret_pts_total'] += w_ret_pts_total
        d_s['ret_bp_conv']   += w_bp_conv
        d_s['ret_bp_opps']   += w_bp_opps
        d_s['ret_gms_played'] += w_ret_gms_played
        d_s['ret_gms_won']   += w_ret_gms_won

    #if loser is returning
    l_ret_pts_won   = row['w_svpt'] - row['w_1stWon'] - row['w_2ndWon']
    l_ret_pts_total = row['w_svpt']
    l_bp_opps       = row['w_bpFaced']
    l_bp_conv       = row['w_bpFaced'] - row['w_bpSaved']
    l_ret_gms_played = row['w_SvGms']
    l_ret_gms_won    = l_bp_conv

    for d_o, d_s in [(l_o, l_s)]:
        d_o['ret_pts_won']   += l_ret_pts_won
        d_o['ret_pts_total'] += l_ret_pts_total
        d_o['ret_bp_conv']   += l_bp_conv
        d_o['ret_bp_opps']   += l_bp_opps
        d_o['ret_gms_played'] += l_ret_gms_played
        d_o['ret_gms_won']   += l_ret_gms_won

        d_s['ret_pts_won']   += l_ret_pts_won
        d_s['ret_pts_total'] += l_ret_pts_total
        d_s['ret_bp_conv']   += l_bp_conv
        d_s['ret_bp_opps']   += l_bp_opps
        d_s['ret_gms_played'] += l_ret_gms_played
        d_s['ret_gms_won']   += l_ret_gms_won

    #total points
    total_pts_w = row['w_1stWon'] + row['w_2ndWon'] + w_ret_pts_won
    total_pts_l = row['l_1stWon'] + row['l_2ndWon'] + l_ret_pts_won
    total_pts_total = row['w_svpt'] + row['l_svpt']

    for d_o, d_s in [(w_o, w_s)]:
        d_o['total_pts_won']   += total_pts_w
        d_o['total_pts_total'] += total_pts_total
        d_s['total_pts_won']   += total_pts_w
        d_s['total_pts_total'] += total_pts_total

    for d_o, d_s in [(l_o, l_s)]:
        d_o['total_pts_won']   += total_pts_l
        d_o['total_pts_total'] += total_pts_total
        d_s['total_pts_won']   += total_pts_l
        d_s['total_pts_total'] += total_pts_total


In [ ]:
feature_rows = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    surface = row['surface']
    date    = row['tourney_date']

    w_id = int(row['winner_id'])
    l_id = int(row['loser_id'])

    #features before match
    w_feats = compute_player_features(
        w_id, surface,
        rank=row['winner_rank'],
        age=row['winner_age'],
        height=row['winner_ht']
    )
    l_feats = compute_player_features(
        l_id, surface,
        rank=row['loser_rank'],
        age=row['loser_age'],
        height=row['loser_ht']
    )

    #winner perspective
    diff = {f: w_feats[f] - l_feats[f] for f in w_feats.keys()}
    diff['tourney_date'] = date
    diff['y'] = 1
    feature_rows.append(diff)

    #loser perspective
    diff_neg = {f: -diff[f] for f in w_feats.keys()}
    diff_neg['tourney_date'] = date
    diff_neg['y'] = 0
    feature_rows.append(diff_neg)

    #update history with match
    update_player_after_match(row)

features_df = pd.DataFrame(feature_rows)
print(features_df.shape)
features_df.head()


In [ ]:
features_df['year'] = features_df['tourney_date'].dt.year

train_mask = features_df['year'] <= 2023
test_mask  = features_df['year'] == 2024

train = features_df[train_mask].copy()
test  = features_df[test_mask].copy()

feature_cols = [c for c in features_df.columns
                if c not in ['y', 'tourney_date', 'year']]

X_train = train[feature_cols].values
y_train = train['y'].values
X_test  = test[feature_cols].values
y_test  = test['y'].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_s, y_train)

y_pred  = log_reg.predict(X_test_s)
y_proba = log_reg.predict_proba(X_test_s)[:, 1]

print("2024 Accuracy:", accuracy_score(y_test, y_pred))
print("2024 ROC AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coef': log_reg.coef_[0]
}).sort_values(by='coef', key=lambda s: s.abs(), ascending=False)
coef_df.head(20)

In [ ]:
# Build a name → player_id lookup
player_name_to_id = {}

for _, row in df.iterrows():
    player_name_to_id[row['winner_name']] = int(row['winner_id'])
    player_name_to_id[row['loser_name']]  = int(row['loser_id'])

def get_player_id(name):
    return player_name_to_id.get(name, None)


In [ ]:
def make_match_features(p1_id, p2_id, surface,
                        p1_rank=5000, p2_rank=5000,
                        p1_age=25,  p2_age=25,
                        p1_height=185, p2_height=185):

    # Historical feature vectors
    p1_feats = compute_player_features(p1_id, surface, p1_rank, p1_age, p1_height)
    p2_feats = compute_player_features(p2_id, surface, p2_rank, p2_age, p2_height)

    # Build the difference vector
    diff = {f: p1_feats[f] - p2_feats[f] for f in p1_feats.keys()}

    return diff


In [ ]:
def predict_winner(player1, player2, surface):

    # Convert names to IDs if needed
    p1_id = player1 if isinstance(player1, int) else get_player_id(player1)
    p2_id = player2 if isinstance(player2, int) else get_player_id(player2)

    if p1_id is None:
        return f"Unknown player name: {player1}"
    if p2_id is None:
        return f"Unknown player name: {player2}"

    # Auto-fill metadata from your lookup tables
    p1_rank  = player_rank.get(p1_id, 5000)
    p2_rank  = player_rank.get(p2_id, 5000)

    p1_age   = player_age.get(p1_id, 28)     # fallback if very old matches missing age
    p2_age   = player_age.get(p2_id, 28)

    p1_height = player_height.get(p1_id, 185)
    p2_height = player_height.get(p2_id, 185)

    # Build feature vector (P1 - P2)
    feats = make_match_features(
        p1_id, p2_id, surface,
        p1_rank=p1_rank, p2_rank=p2_rank,
        p1_age=p1_age, p2_age=p2_age,
        p1_height=p1_height, p2_height=p2_height
    )

    # Convert to array in correct order
    x = np.array([[feats[f] for f in feature_cols]])
    x_scaled = scaler.transform(x)

    prob = log_reg.predict_proba(x_scaled)[0, 1]

    return {
        'player1': player1,
        'player2': player2,
        'surface': surface,
        'p(Player1 wins)': float(prob),
        'p(Player2 wins)': float(1 - prob),
        'predicted_winner': player1 if prob >= 0.5 else player2
    }


In [ ]:
predict_winner("Carlos Alcaraz", "Roger Federer", "hard")


In [ ]:
df.head